In [ ]:
%pip install -U langgraph langchain-openai python-dotenv

In [ ]:
%pip install -U langsmith

In [ ]:
%pip install -U langsmith langchain-openai

In [ ]:
import os
import json
import re
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, START, END
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
from getpass import getpass
from langsmith import Client, tracing_context
from langchain_openai import ChatOpenAI

from langsmith import Client

client = Client(
    api_key="lsv2_",
    api_url="https://api.smith.langchain.com"
)

try:
    next(client.list_projects(limit=1), None)
    print("Conexão com LangSmith OK!")
except Exception as e:
    print("Erro:", str(e))

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key="sk-"
)
# Definir a estrutura de dados do candidato
class Candidate(TypedDict):
    name: str
    resume: str
    skills: List[str]
    experience: int
    education_level: str
    score: float
    feedback: str

# Definir o estado do fluxo de trabalho
class State(TypedDict):
    candidates: List[Candidate]
    job_requirements: Dict[str, any]
    result: str



Conexão com LangSmith OK!


In [2]:

import re

def extract_information_from_resume(resume_text: str) -> dict:
    """Extrai informações do currículo usando regex."""
    info = {}

    # Extrair nome
    name_match = re.search(
        r'Nome:\s*([^\n]+)', resume_text, re.IGNORECASE
    )
    info['name'] = name_match.group(1).strip() if name_match else ''

    # Extrair experiência
    exp_match = re.search(
        r'(\d+)\s*anos? de experiência',
        resume_text, re.IGNORECASE
    )
    info['experience'] = int(exp_match.group(1)) if exp_match else 0

    # Extrair educação
    edu_patterns = [
        r'\bp[oó]s[-\s]*doutorand[oa]\b(?:\s+em\s+[^.,\n]+)?',
        r'\bp[oó]s[-\s]*doutorado\b(?=[^.\n]*\bem andamento\b)[^.\n]*',
        r'\bp[oó]s[-\s]*doutorado\b(?:\s+em\s+[^.,\n]+)?',
        r'\bdoutorand[oa]\b(?:\s+em\s+[^.,\n]+)?',
        r'\bdoutorado\b(?=[^.\n]*\bem andamento\b)[^.\n]*',
        r'\bdoutorado em [^.,\n]+',
        r'\bdoutor(?:a)? em [^.,\n]+',
        r'\bmestrand[oa]\b(?:\s+em\s+[^.,\n]+)?',
        r'\bmestrado\b(?=[^.\n]*\bem andamento\b)[^.\n]*',
        r'\bmestrado em [^.,\n]+',
        r'\bmestre em [^.,\n]+',
        r'\bgraduação em [^.,\n]+',
        r'\b(?:rec[eé]m[-\s]?)?formad[ao] em [^.,\n]+'
    ]

    info['education_level'] = ''

    for pattern in edu_patterns:
        edu_match = re.search(
            pattern, resume_text, re.IGNORECASE
        )

        if edu_match:
            info['education_level'] = edu_match.group(0).strip()
            break

    # Extrair habilidades
    skills = []

    skill_patterns = [
        r'proficiente em ([^.\n]+)',
        r'conhecimento em ([^.\n]+)',
        r'especialista em ([^.\n]+)',
        r'experiência (?:com|em) ([^.\n]+)'
    ]

    for pattern in skill_patterns:
        skill_match = re.search(
            pattern, resume_text, re.IGNORECASE
        )

        if skill_match:
            texto = skill_match.group(1)

            skills.extend(
                s.strip()
                for s in re.split(r',|\s+e\s+', texto)
                if s.strip()
            )

    info['skills'] = list(dict.fromkeys(skills))

    return info


In [3]:

import re

def score_education(education_level: str) -> float:
    """Calcula a pontuação acadêmica do candidato."""

    edu_patterns = [
        r'\bp[oó]s[-\s]*doutorand[oa]\b',
        r'\bp[oó]s[-\s]*doutorado\b(?=[^.,\n]*\bem andamento\b)',
        r'\bp[oó]s[-\s]*doutorado\b(?=[^.,\n]*\bconclu[ií]d[oa]\b)',
        r'\bp[oó]s[-\s]*doutorado\b',
        r'\bdoutorand[oa]\b',
        r'\bdoutorado\b(?=[^.,\n]*\bem andamento\b)',
        r'\bdoutorado\b',
        r'\bdoutor(?:a)?\b',
        r'\bmestrand[oa]\b',
        r'\bmestrado\b(?=[^.,\n]*\bem andamento\b)',
        r'\bmestrado\b',
        r'\bmestre\b',
        r'\bgraduação\b',
        r'\b(?:rec[eé]m[-\s]?)?formad[ao] em\b'
    ]

    # Pontuação correspondente a cada padrão
    edu_scores = [
        [8.5],                  # Pós-doutorando
        [8.5],                  # Pós-doutorado em andamento
        [10.0],                  # Pós-doutorado concluído
        [10.0],                  # Pós-doutorado
        [7.5],                   # Doutorando
        [7.5],                   # Doutorado em andamento
        [10.0],                  # Doutorado concluído
        [10.0],                  # Doutor
        [4.5],                   # Mestrando
        [4.5],                   # Mestrado em andamento
        [6.0],                   # Mestrado concluído
        [6.0],                   # Mestre
        [5.0],                   # Graduação
        [5.0]                    # Formado
    ]

    # Verificar formação e somar as pontuações
    for pattern, scores in zip(edu_patterns, edu_scores):
        edu_match = re.search(
            pattern, education_level or "", re.IGNORECASE
        )

        if edu_match:
            return sum(scores)

    return 0.0


In [4]:
candidates = [
    {
        "resume": """
        Nome: João Silva
        Graduação em Engenharia de Software
        5 anos de experiência
        Proficiente em Python, SQL e Machine Learning.
        """
    },
    {
        "resume": """
        Nome: Maria Souza
        Mestrado em Ciência da Computação
        3 anos de experiência
        Conhecimento em Python, LangChain e LangGraph.
        """
    },
    {
        "resume": """
        Nome: Thay Oliveira
        Doutorado em Inteligência Artificial
        4 anos de experiência
        Especialista em Machine Learning e Deep Learning.
        """
    }
]

In [5]:
# Definir a estrutura de dados do candidato
class Candidate(TypedDict):
    name: str
    resume: str
    skills: List[str]
    experience: int
    education_level: str
    score: float
    feedback: str

# Definir o estado do fluxo de trabalho
class State(TypedDict):
    candidates: List[Candidate]
    job_requirements: Dict[str, any]
    result: str


import pandas as pd
from IPython.display import display

resultados = []

for candidate in candidates:
    info = extract_information_from_resume(candidate["resume"])

    resultados.append({
        "Nome": info["name"],
        "Formação": info["education_level"],
        "Nota acadêmica": score_education(info["education_level"]),
        "Experiência (anos)": info["experience"]
    })



In [6]:
def evaluate_candidate(state: State) -> State:
    """Avalia os candidatos com base nos requisitos."""
    updated_candidates = []
    
    for candidate in state['candidates']:
        try:
            # Sistema de pontuação direto
            score = 0
            feedback_points = []
            
            # Avaliar habilidades (4 pontos)
            required_skills = state['job_requirements']['skills_required']
            matching_skills = []
            for req_skill in required_skills:
                for candidate_skill in candidate['skills']:
                    if req_skill.lower() in candidate_skill.lower():
                        matching_skills.append(req_skill)
                        break
            
            skill_score = len(set(matching_skills)) / len(required_skills) * 4
            score += skill_score
            feedback_points.append(f"Possui {len(set(matching_skills))} de {len(required_skills)} habilidades requeridas")
            
            # Avaliar experiência (3 pontos)
            min_exp = state['job_requirements']['minimum_experience']
            if candidate['experience'] >= min_exp:
                score += 3
                feedback_points.append(f"Experiência adequada ({candidate['experience']} anos)")
            else:
                feedback_points.append(f"Experiência abaixo do requisito ({candidate['experience']} anos)")
            
            # Avaliar educação (3 pontos)
            if candidate['education_level']:
                score += 3
                feedback_points.append(f"Possui formação adequada: {candidate['education_level']}")
            
            # Atualizar candidato
            candidate['score'] = min(score, 10)  # Limita a 10 pontos
            candidate['feedback'] = ". ".join(feedback_points)
            
        except Exception as e:
            candidate['score'] = 0.0
            candidate['feedback'] = f"Erro na avaliação: {str(e)}"
        
        updated_candidates.append(candidate)
    
    state['candidates'] = updated_candidates
    return state

def rank_candidates(state: State) -> State:
    """Classifica os candidatos por pontuação."""
    try:
        state['candidates'].sort(key=lambda x: (float(x.get('score', 0)), x.get('experience', 0)), reverse=True)
    except Exception as e:
        print(f"Erro ao classificar candidatos: {e}")
    return state

In [7]:
import re
import unicodedata

def normalizar_texto(texto):
    texto = unicodedata.normalize("NFKD", str(texto or ""))
    return "".join(
        c for c in texto
        if not unicodedata.combining(c)
    ).casefold()


def calcular_nota_skills(candidate, skills_required):
    skills = candidate.get("skills") or []

    if isinstance(skills, str):
        skills = [skills]

    texto = normalizar_texto(
        candidate.get("resume", "") + " " + " ".join(skills)
    )

    skills_atendidas = []

    for skill in skills_required:
        padrao = rf"(?<!\w){re.escape(normalizar_texto(skill))}(?!\w)"

        if re.search(padrao, texto):
            skills_atendidas.append(skill)

    skills_ausentes = [
        skill for skill in skills_required
        if skill not in skills_atendidas
    ]

    nota = (
        10.0 * len(skills_atendidas) / len(skills_required)
        if skills_required else None
    )

    return nota, skills_atendidas, skills_ausentes

In [8]:

def generate_result(state: State) -> State:
    """Gera relatório detalhado de classificação em Markdown."""

    candidates = state.get("candidates") or []
    req = state.get("job_requirements") or {}

    if not candidates:
        state["result"] = "## Nenhum candidato disponível."
        return state

    report = []
    skills_required = req.get("skills_required") or []

    report.extend([
        "# RELATÓRIO DE CLASSIFICAÇÃO DE CANDIDATOS",
        "",
        "## Requisitos da Vaga",
        "",
        f"**Habilidades:** {', '.join(skills_required)}",
        "",
        f"**Experiência mínima:** {req.get('minimum_experience', 0)} anos",
        "",
        f"**Formação exigida:** {req.get('education_level', 'Não informado')}",
        "",
        f"**Atributos desejados:** {', '.join(req.get('desired_attributes') or [])}",
        "",
        "---",
        "",
        "## Ranking Completo",
        "",
        "| Posição | Candidato | Nota Skills | Nota Acadêmica | Nota Final |",
        "|:---:|---|---:|---:|---:|"
    ])

    avaliacoes = []

    for i, candidate in enumerate(candidates, 1):

        nome = candidate.get("name") or f"Candidato {i}"
        formacao = candidate.get("education_level") or ""
        nota_final = float(candidate.get("score") or 0)

        nota_academica = score_education(formacao)

        nota_skills, atendidas, ausentes = calcular_nota_skills(
            candidate, skills_required
        )

        avaliacoes.append({
            "candidate": candidate,
            "nome": nome,
            "nota_final": nota_final,
            "nota_academica": nota_academica,
            "nota_skills": nota_skills,
            "atendidas": atendidas,
            "ausentes": ausentes
        })

        texto_skills = (
            f"{nota_skills:.1f}/10"
            if nota_skills is not None else "N/A"
        )

        report.append(
            f"| {i} | {nome} | **{texto_skills}** | "
            f"{nota_academica:.1f} pontos | "
            f"**{nota_final:.1f}/10** |"
        )

    report.extend([
        "",
        "---",
        "",
        "## Avaliação Detalhada dos Candidatos",
        ""
    ])

    for i, avaliacao in enumerate(avaliacoes, 1):

        candidate = avaliacao["candidate"]
        nome = avaliacao["nome"]

        formacao = candidate.get("education_level") or "Não informado"
        experiencia = float(candidate.get("experience") or 0)
        skills = candidate.get("skills") or []
        feedback = candidate.get("feedback") or "Não disponível"

        if isinstance(skills, str):
            skills = [skills]

        nota_final = avaliacao["nota_final"]
        nota_academica = avaliacao["nota_academica"]
        nota_skills = avaliacao["nota_skills"]

        atendidas = avaliacao["atendidas"]
        ausentes = avaliacao["ausentes"]

        texto_skills = (
            f"{nota_skills:.1f}/10"
            if nota_skills is not None else "N/A"
        )

        report.extend([
            f"### {i}º — {nome}",
            "",
            "| Critério | Resultado |",
            "|---|---|",
            f"| Formação | {formacao} |",
            f"| Nota Acadêmica | **{nota_academica:.1f} pontos** |",
            f"| Experiência | {experiencia:g} anos |",
            f"| Nota de Skills | **{texto_skills}** |",
            f"| Nota Final | **{nota_final:.1f}/10** |",
            "",
            "#### Avaliação das Habilidades",
            "",
            "| Habilidade Exigida | Situação | Pontos |",
            "|---|:---:|---:|"
        ])

        pontos_por_skill = (
            10.0 / len(skills_required)
            if skills_required else 0.0
        )

        for skill in skills_required:

            encontrada = skill in atendidas

            status = "Atendida" if encontrada else "Não identificada"
            pontos = pontos_por_skill if encontrada else 0.0

            report.append(
                f"| {skill} | {status} | {pontos:.2f} |"
            )

        report.extend([
            "",
            f"**Nota total de Skills: {texto_skills}**",
            "",
            f"**Skills identificadas:** {', '.join(skills) or 'Nenhuma'}",
            "",
            f"**Skills exigidas atendidas:** {', '.join(atendidas) or 'Nenhuma'}",
            "",
            f"**Skills não identificadas:** {', '.join(ausentes) or 'Nenhuma'}",
            "",
            "#### Feedback da Avaliação",
            "",
            str(feedback),
            "",
            "---",
            ""
        ])

    state["result"] = "\n".join(report)

    return state
def extract_information(state: State) -> dict:
    """Extrai informações de todos os candidatos."""
    updated_candidates = []

    for candidate in state["candidates"]:
        info = extract_information_from_resume(
            candidate["resume"]
        )

        if not isinstance(info, dict):
            raise TypeError(
                "extract_information_from_resume deve retornar um dicionário"
            )

        updated_candidates.append({
            **candidate,
            **info,
            "score": 0.0,
            "feedback": ""
        })

    return {"candidates": updated_candidates}

# Construir o workflow
workflow = StateGraph(State)

# Adicionar nós
workflow.add_node("extract_information", extract_information)
workflow.add_node("evaluate_candidate", evaluate_candidate)
workflow.add_node("rank_candidates", rank_candidates)
workflow.add_node("generate_result", generate_result)

# Definir as arestas
workflow.add_edge(START, "extract_information")
workflow.add_edge("extract_information", "evaluate_candidate")
workflow.add_edge("evaluate_candidate", "rank_candidates")
workflow.add_edge("rank_candidates", "generate_result")
workflow.add_edge("generate_result", END)

# Compilar o grafo
app = workflow.compile()

def process_candidates(candidates: List[Candidate], job_requirements: Dict[str, any]) -> str:
    """Processa os candidatos e retorna o resultado."""
    try:
        initial_state = State(
            candidates=candidates,
            job_requirements=job_requirements,
            result=""
        )
        
        final_state = app.invoke(initial_state)
        return final_state.get('result', 'Erro: Resultado não disponível')
    except Exception as e:
        return f"Erro durante o processamento: {str(e)}"

# Exemplo de uso
if __name__ == "__main__":
    # Exemplo de candidatos
    candidates = [
        {
            "resume": """
            Nome: Thay Jr
            
            Thay Jr está estudando consistentemente o curso compreto da Scoras Academy para se expecializar em IA. 
            El é proficiente em Python, Entende profundamente de LangGrapg, PydanticAI, SLMs e LLMs e tem experiência com bancos de dados SQL. 
            Ele é engenheiro mecatrônico, mestre em informática aplicada edoutor em engenharia elétrica, pós doutorando em engenharia de teleinformática.
            Está com 1 anos de experiência em desenvolvimento de software.
            """
        },
        {
            "resume": """
            Nome: João Souza
            
            João é engenheiro de software com 8 anos de experiência. 
            Especialista em Java, SQL e Desenvolvimento Web. 
            Possui mestrado em Engenharia de Software.
            Tem experiência em liderança de equipes e desenvolvimento de sistemas de grande escala.
            """
        },
        {
            "resume": """
            Nome: Ana Pereira
            
            Ana é recém-formada em Sistemas de Informação. 
            Possui conhecimento em Python, HTML e CSS. 
            Participou de projetos acadêmicos relacionados a desenvolvimento web.
            Trabalhou em projetos de desenvolvimento web e está familiarizada com metodologias ágeis.
            """
        }
    ]

    # Requisitos da vaga
    job_requirements = {
        "skills_required": ["Python", "SQL", "Desenvolvimento Web", "LangGraph", "PydanticAI", "LLMs", "SLMs"],
        "minimum_experience": 3,
        "education_level": "Graduação em área relacionada",
        "desired_attributes": ["Trabalho em equipe", "Metodologias Ágeis"]
    }


from IPython.display import display, Markdown
    # Processar os candidatos
result = process_candidates(candidates, job_requirements)

display(Markdown(result))

 



# RELATÓRIO DE CLASSIFICAÇÃO DE CANDIDATOS

## Requisitos da Vaga

**Habilidades:** Python, SQL, Desenvolvimento Web, LangGraph, PydanticAI, LLMs, SLMs

**Experiência mínima:** 3 anos

**Formação exigida:** Graduação em área relacionada

**Atributos desejados:** Trabalho em equipe, Metodologias Ágeis

---

## Ranking Completo

| Posição | Candidato | Nota Skills | Nota Acadêmica | Nota Final |
|:---:|---|---:|---:|---:|
| 1 | João Souza | **2.9/10** | 6.0 pontos | **7.1/10** |
| 2 | Thay Jr | **7.1/10** | 8.5 pontos | **5.9/10** |
| 3 | Ana Pereira | **2.9/10** | 5.0 pontos | **3.6/10** |

---

## Avaliação Detalhada dos Candidatos

### 1º — João Souza

| Critério | Resultado |
|---|---|
| Formação | mestrado em Engenharia de Software |
| Nota Acadêmica | **6.0 pontos** |
| Experiência | 8 anos |
| Nota de Skills | **2.9/10** |
| Nota Final | **7.1/10** |

#### Avaliação das Habilidades

| Habilidade Exigida | Situação | Pontos |
|---|:---:|---:|
| Python | Não identificada | 0.00 |
| SQL | Atendida | 1.43 |
| Desenvolvimento Web | Atendida | 1.43 |
| LangGraph | Não identificada | 0.00 |
| PydanticAI | Não identificada | 0.00 |
| LLMs | Não identificada | 0.00 |
| SLMs | Não identificada | 0.00 |

**Nota total de Skills: 2.9/10**

**Skills identificadas:** Java, SQL, Desenvolvimento Web, liderança de equipes, desenvolvimento de sistemas de grande escala

**Skills exigidas atendidas:** SQL, Desenvolvimento Web

**Skills não identificadas:** Python, LangGraph, PydanticAI, LLMs, SLMs

#### Feedback da Avaliação

Possui 2 de 7 habilidades requeridas. Experiência adequada (8 anos). Possui formação adequada: mestrado em Engenharia de Software

---

### 2º — Thay Jr

| Critério | Resultado |
|---|---|
| Formação | pós doutorando em engenharia de teleinformática |
| Nota Acadêmica | **8.5 pontos** |
| Experiência | 1 anos |
| Nota de Skills | **7.1/10** |
| Nota Final | **5.9/10** |

#### Avaliação das Habilidades

| Habilidade Exigida | Situação | Pontos |
|---|:---:|---:|
| Python | Atendida | 1.43 |
| SQL | Atendida | 1.43 |
| Desenvolvimento Web | Não identificada | 0.00 |
| LangGraph | Não identificada | 0.00 |
| PydanticAI | Atendida | 1.43 |
| LLMs | Atendida | 1.43 |
| SLMs | Atendida | 1.43 |

**Nota total de Skills: 7.1/10**

**Skills identificadas:** Python, Entende profundamente de LangGrapg, PydanticAI, SLMs, LLMs, tem experiência com bancos de dados SQL, bancos de dados SQL

**Skills exigidas atendidas:** Python, SQL, PydanticAI, LLMs, SLMs

**Skills não identificadas:** Desenvolvimento Web, LangGraph

#### Feedback da Avaliação

Possui 5 de 7 habilidades requeridas. Experiência abaixo do requisito (1 anos). Possui formação adequada: pós doutorando em engenharia de teleinformática

---

### 3º — Ana Pereira

| Critério | Resultado |
|---|---|
| Formação | recém-formada em Sistemas de Informação |
| Nota Acadêmica | **5.0 pontos** |
| Experiência | 0 anos |
| Nota de Skills | **2.9/10** |
| Nota Final | **3.6/10** |

#### Avaliação das Habilidades

| Habilidade Exigida | Situação | Pontos |
|---|:---:|---:|
| Python | Atendida | 1.43 |
| SQL | Não identificada | 0.00 |
| Desenvolvimento Web | Atendida | 1.43 |
| LangGraph | Não identificada | 0.00 |
| PydanticAI | Não identificada | 0.00 |
| LLMs | Não identificada | 0.00 |
| SLMs | Não identificada | 0.00 |

**Nota total de Skills: 2.9/10**

**Skills identificadas:** Python, HTML, CSS

**Skills exigidas atendidas:** Python, Desenvolvimento Web

**Skills não identificadas:** SQL, LangGraph, PydanticAI, LLMs, SLMs

#### Feedback da Avaliação

Possui 1 de 7 habilidades requeridas. Experiência abaixo do requisito (0 anos). Possui formação adequada: recém-formada em Sistemas de Informação

---
